### Truncation thresholds tutorial

This notebooks provides the code to visualize what modes are being truncated based on the input threshold.

In [ ]:
import numpy as np
import matplotlib.gridspec as gridspec
from lsst.ts.ofc import OFC, OFCData, SensitivityMatrix
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

%load_ext autoreload
%autoreload 2

%matplotlib inline

Initialize below the sensitivity matrix of interest. It returns the rescaled (normalized) sensitivity matrix. You will need to modify this cell if you want to use other detectors than the ComCam ones. You can also modify it by setting ofc_data.zn_selected, in case not all zernikes will be used. When trying different normalization weights, you will need to re-run this notebook. 

In [ ]:
ofc_data = OFCData('comcam')
dz_sensitivity_matrix = SensitivityMatrix(ofc_data)
sensor_name_list = [        
        "R22_S00",
        "R22_S01",
        "R22_S02",
        "R22_S10",
        "R22_S11",
        "R22_S12",
        "R22_S20",
        "R22_S21",
        "R22_S22",
]

field_angles = [
    ofc_data.sample_points[sensor] for sensor in sensor_name_list
]

dz_sensitivity_matrix = SensitivityMatrix(ofc_data)
sensitivity_matrix = dz_sensitivity_matrix.evaluate(
    field_angles, 0.0
)

sensitivity_matrix = sensitivity_matrix[:, ofc_data.zn_idx, :]

size = sensitivity_matrix.shape[2]
sensitivity_matrix = sensitivity_matrix.reshape((-1, size))

sensitivity_matrix = sensitivity_matrix[..., ofc_data.dof_idx]

normalization_matrix = np.diag(
    ofc_data.normalization_weights[ofc_data.dof_idx]
)
sensitivity_matrix = sensitivity_matrix @ normalization_matrix

Below the characteristic modes given the normalization weights used are plotted on the left. On the right, is the singular values for each of them along with the cutoff. One can set the cutoff by number or by value.

In [ ]:
labels = ['M2 dz', 'M2 dx', 'M2 dy', 'M2 rx', 'M2 ry',
         'cam dz', 'cam dx', 'cam dy', 'cam rx', 'cam ry',
         '$B_{{1,1}}$', '$B_{{1,2}}$', '$B_{{1,3}}$', '$B_{{1,4}}$', '$B_{{1,5}}$',
         '$B_{{1,6}}$', '$B_{{1,7}}$', '$B_{{1,8}}$', '$B_{{1,9}}$', '$B_{{1,10}}$',
         '$B_{{1,11}}$', '$B_{{1,12}}$', '$B_{{1,13}}$', '$B_{{1,14}}$', '$B_{{1,15}}$',
         '$B_{{1,16}}$', '$B_{{1,17}}$', '$B_{{1,18}}$', '$B_{{1,19}}$', '$B_{{1,20}}$',
         '$B_{{2,1}}$', '$B_{{2,2}}$', '$B_{{2,3}}$', '$B_{{2,4}}$', '$B_{{2,5}}$',
         '$B_{{2,6}}$', '$B_{{2,7}}$', '$B_{{2,8}}$', '$B_{{2,9}}$', '$B_{{2,10}}$',
         '$B_{{2,11}}$', '$B_{{2,12}}$', '$B_{{2,13}}$', '$B_{{2,14}}$', '$B_{{2,15}}$',
         '$B_{{2,16}}$', '$B_{{2,17}}$', '$B_{{2,18}}$', '$B_{{2,19}}$', '$B_{{2,20}}$'
         ]
u, s, vh = np.linalg.svd(sensitivity_matrix[:,  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]], full_matrices=True)
v = vh.T

fig = plt.figure(figsize=(9, 4), dpi=600)
gs = gridspec.GridSpec(1,2, width_ratios=[1.2,1])
a0 = plt.subplot(gs[0])
a1 = plt.subplot(gs[1])

r = a0.imshow(v, cmap = 'seismic', vmin = -1, vmax = 1)

a0.set_yticks([0, 5, 10], [labels[0], labels[5], labels[10]])
a0.set_xticks([0, 10], ['$v_0$', '$v_{10}$'])
fig.colorbar(r)
a0.tick_params(direction='in', which='both', labelsize=12) # Apply to both major and minor ticks
a0.xaxis.set_minor_locator(ticker.MultipleLocator(5))
a0.yaxis.set_minor_locator(ticker.MultipleLocator(5))
a0.set_xlabel('Characteristic modes')
a0.grid(alpha=0.25)
a0.tick_params(direction = 'in', labelsize=12)


a1.semilogy(s, '.-', markersize=4)
#truncate_index = 25
#cutoff =  0.85 * s[truncate_index - 1] / np.max(s)
#print(cutoff)
cutoff = 1.e-4
a1.semilogy(np.max(s)*cutoff*np.ones(45))
a1.set_xticks([0, 10], ['$\sigma_0$', '$\sigma_{10}$'])
a1.set_xlabel('Singular values')
a1.grid(alpha=0.25)
a1.tick_params(direction='in', which='both', labelsize=12) # Apply to both major and minor ticks

a1.xaxis.set_minor_locator(ticker.AutoMinorLocator())  # Auto set minor ticks on x-axis